# 08 - Porter's Diamond Interpretation

Research question: why do states perform the way they do, seen through Porter's Diamond?
Reads Version 1.0 outputs and interprets each determinant. Every claim is backed by the
data. Where official data is missing, the limit is stated, not filled with guesswork.

## Setup

Load the Version 1.0 outputs and the income data used only to describe Demand Conditions.
Nothing is changed. The report is built section by section.

In [1]:
import sys
from pathlib import Path
import pandas as pd

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from src import scoring

reports_dir = root / "version2" / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

scores = pd.read_csv(root / "results" / "indicator_scores.csv", index_col=0)
dimensions = pd.read_csv(root / "results" / "dimension_scores.csv", index_col=0)
index = pd.read_csv(root / "results" / "competitiveness_index.csv", index_col=0)
ranked = index[index["Rank"].notna()].sort_values("Rank")

# income data, used only to describe Demand Conditions (not part of the index)
pcn = pd.read_csv(root / "data" / "processed" / "per_capita_nsdp.csv")
tn = pd.read_csv(root / "data" / "processed" / "total_nsdp.csv")

print("ranked:", len(ranked), "| FC & Related dims:", list(dimensions.columns))
print("income files loaded:", pcn.shape, tn.shape)

ranked: 33 | FC & Related dims: ['factor_conditions', 'related_supporting']
income files loaded: (470, 3) (470, 3)


## Factor Conditions — evidence

Which states are strongest and weakest on Factor Conditions, which indicators states do
best and worst on, and how closely this dimension tracks the final score.

In [2]:
fc = scoring.FACTOR_CONDITIONS

fc_dim = dimensions.loc[ranked.index, "factor_conditions"].sort_values(ascending=False)
fc_avg = scores.loc[ranked.index, fc].mean().sort_values(ascending=False)
fc_corr = dimensions.loc[ranked.index, "factor_conditions"].corr(ranked["competitiveness_score"])

print("Strongest states:"); print(fc_dim.head(5).round(3).to_string())
print("\nWeakest states:"); print(fc_dim.tail(5).round(3).to_string())
print("\nNational average by indicator (higher = states do better):")
print(fc_avg.round(3).to_string())
print(f"\nLink between Factor Conditions and the final score: {fc_corr:.2f}")

Strongest states:
state
Delhi             0.565
Tamil Nadu        0.561
Andhra Pradesh    0.529
Maharashtra       0.518
Telangana         0.511

Weakest states:
state
Uttar Pradesh        0.308
Bihar                0.294
Nagaland             0.292
Meghalaya            0.267
Arunachal Pradesh    0.261

National average by indicator (higher = states do better):
td_losses            0.662
unemployment_rate    0.625
life_expectancy      0.573
ger                  0.472
cd_ratio             0.382
road_density         0.195
per_capita_power     0.067

Link between Factor Conditions and the final score: 0.83


## Related & Supporting Industries — evidence

Strongest and weakest states on this dimension, the indicator averages, how closely it
tracks the final score, and how widely it spreads states.

In [3]:
rs = scoring.RELATED_SUPPORTING

rs_dim = dimensions.loc[ranked.index, "related_supporting"].sort_values(ascending=False)
rs_avg = scores.loc[ranked.index, rs].mean().sort_values(ascending=False)
rs_corr = dimensions.loc[ranked.index, "related_supporting"].corr(ranked["competitiveness_score"])
fc_std = dimensions.loc[ranked.index, "factor_conditions"].std()
rs_std = dimensions.loc[ranked.index, "related_supporting"].std()

print("Strongest states:"); print(rs_dim.head(5).round(3).to_string())
print("\nWeakest states:"); print(rs_dim.tail(5).round(3).to_string())
print("\nNational average by indicator:"); print(rs_avg.round(3).to_string())
print(f"\nLink with the final score: {rs_corr:.2f}")
print(f"Spread: Factor Conditions {fc_std:.2f}, Related & Supporting {rs_std:.2f}")

Strongest states:
state
Gujarat       0.829
Goa           0.825
Tamil Nadu    0.681
Puducherry    0.677
Haryana       0.580

Weakest states:
state
Manipur                      0.143
Mizoram                      0.136
Andaman & Nicobar Islands    0.112
Arunachal Pradesh            0.102
Nagaland                     0.072

National average by indicator:
msme_density           0.451
manufacturing_share    0.366
factory_density        0.355
investment_rate        0.312

Link with the final score: 0.93
Spread: Factor Conditions 0.08, Related & Supporting 0.21


## Demand Conditions — proxy evidence

Demand Conditions is not part of the index. It is described here using two official income
measures as proxies: total NSDP (size of the state economy) and per-capita NSDP (income
level). These are shown for context only and do not affect the index.

In [4]:
from src.entities import clean_entity

def load_nsdp(name, year="2022-23"):
    d = pd.read_csv(root / "data" / "processed" / f"{name}.csv")
    d["entity"] = d["state"].map(clean_entity)
    return d[(d["year"] == year) & d["entity"].notna()].set_index("entity")["value"]

pc_income = load_nsdp("per_capita_nsdp").reindex(ranked.index).dropna()
market_size = load_nsdp("total_nsdp").reindex(ranked.index).dropna()

print("Per-capita NSDP 2022-23 (income level) — top 5:")
print(pc_income.sort_values(ascending=False).head(5).round(0).to_string())
print("\nTotal NSDP 2022-23 (market size) — top 5:")
print(market_size.sort_values(ascending=False).head(5).round(0).to_string())
print(f"\ncorr(per-capita NSDP, score): {pc_income.corr(ranked['competitiveness_score'].reindex(pc_income.index)):.2f}")
print(f"corr(total NSDP, score): {market_size.corr(ranked['competitiveness_score'].reindex(market_size.index)):.2f}")

Per-capita NSDP 2022-23 (income level) — top 5:
state
Goa           532854.0
Sikkim        520466.0
Delhi         417217.0
Chandigarh    408512.0
Telangana     312046.0

Total NSDP 2022-23 (market size) — top 5:
state
Maharashtra      317831779.0
Tamil Nadu       211312784.0
Karnataka        209516100.0
Uttar Pradesh    198644223.0
Gujarat          193845955.0

corr(per-capita NSDP, score): 0.53
corr(total NSDP, score): 0.42


## Overall assessment — evidence

How the two measured dimensions relate to each other, and which states are strong or weak
on both.

In [5]:
fc_dim = dimensions.loc[ranked.index, "factor_conditions"]
rs_dim = dimensions.loc[ranked.index, "related_supporting"]
dim_corr = fc_dim.corr(rs_dim)

both_strong = sorted(set(fc_dim.sort_values(ascending=False).head(10).index) &
                     set(rs_dim.sort_values(ascending=False).head(10).index))
both_weak = sorted(set(fc_dim.sort_values().head(10).index) &
                   set(rs_dim.sort_values().head(10).index))

print(f"correlation between the two dimensions: {dim_corr:.2f}")
print("strong on both (top 10 of each):", both_strong)
print("weak on both (bottom 10 of each):", both_weak)

correlation between the two dimensions: 0.58
strong on both (top 10 of each): ['Goa', 'Puducherry', 'Punjab', 'Tamil Nadu', 'Telangana']
weak on both (bottom 10 of each): ['Arunachal Pradesh', 'Bihar', 'Jammu & Kashmir', 'Manipur', 'Meghalaya', 'Nagaland', 'Uttar Pradesh']
